<a href="https://colab.research.google.com/github/EmilioUcelayLojo/AA3/blob/laboratorios_practicas/lab8/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/lab8/lab8.ipynb)

# Práctica 8: Modelos generativos

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

### NOTA: En <font color='red'>Google Colab</font> hay que instalar los paquetes EN CADA EJECUCIÓN

In [1]:
# Ejemplo de instalación de tensorflow 2.0
#%tensorflow_version 2.x
# !pip3 install tensorflow  # NECESARIO SOLO SI SE EJECUTA EN LOCAL
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# Modelos generativos sobre MNIST

Lo primero que tenemos que hacer es cargar el dataset.

In [2]:
labeled_data = 0.01 # Vamos a usar el etiquetado de sólo el 1% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test), = tf.keras.datasets.mnist.load_data()

indices = np.arange(len(x_train))
np.random.shuffle(indices)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indices[ntrain_data:]]
x_train = x_train[indices[:ntrain_data]]
y_train = y_train[indices[:ntrain_data]]

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

x_train shape: (600, 28, 28)
600 train samples
10000 test samples


In [ ]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)
# Normalizar los valores de los píxeles al rango [0, 1]
x_train = x_train.astype(np.float32) / 255.0
unlabeled_train = unlabeled_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Aplanar las imágenes para que sean vectores de características
x_train = x_train.reshape(x_train.shape[0], -1)
unlabeled_train = unlabeled_train.reshape(unlabeled_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

print(x_train.shape, unlabeled_train.shape, x_test.shape)

(600, 784) (59400, 784) (10000, 784)


## Modelo generativo

Vamos a crear nuestro propio modelo generativo. En clase de teoría has visto muchas versiones distintas:

1. Mezcla de distribuciones de Gaussianas (GMM)
1. Mezcla de distribuciones multinomiales (Naive Bayes)
1. Modelos de Markov ocultos (HMM)

Tal y como se os apunta en teoría, los modelos generativos abordan un problema más general que la clasificación o regresión: aprenden cómo se estructuran y distribuyen los datos de entrada.

En nuestro caso, vamos a modelar los datos de entrada mediante el uso de **Autoencoders**.

# Autoencoders

El autoencoder es un tipo de red que se utiliza para aprender codificaciones eficientes de datos sin etiquetar (lo que se conoce como aprendizaje no supervisado). Es una red que tiene el mismo tamaño en la entrada como en la salida, puesto que el objetivo de la red es reconstruir la entrada con la menor pérdida posible.

Si lo que hacemos es reconstruir la entrada, ¿qué sentido tiene el usar la red? Habitualmente, **la red consta, a su mitad, de una capa con menos elementos que los datos de entrada**. Por tanto, al reconstruir los datos de la entrada a la salida, en esa capa tendremos una versión *comprimida* de la entrada, que contendrá la mayor parte de su información.

Por tanto, podemos dividir un autoencoder en 3 secciones diferentes, tal y como se ve en la siguiente figura:

![](https://drive.google.com/uc?export=view&id=1yxkKZV0J0YplQAGPGJxQ2Z80Ad6L94eu)

1. **Encoder:** es la parte inicial de la red, encargada de comprimir los datos de la entrada.
1. **Code:** es la salida del encoder, contiene la versión *comprimida* de los datos de entrada.
1. **Decoder:** se encarga de, partiendo de la salida del *Encoder*, reconstruir la red.

## Crea tu propio Autoencoder

El diseño del autoencoder es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [ ]:
# TODO: crea tu propio autoencoder

class MiAutoencoder:

    def __init__(self, input_shape):
        # TODO : define el modelo y compílalo

        # Define la arquitectura del encoder
        self.encoder = tf.keras.models.Sequential([
            tf.keras.layers.InputLayer(input_shape=(input_shape,)),  # Entrada
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(64, activation='relu')
        ])

        # Define la arquitectura del decoder
        self.decoder = tf.keras.models.Sequential([
            tf.keras.layers.InputLayer(input_shape=(64,)),  # Código
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(input_shape, activation='sigmoid')
        ])

        # Construye el autoencoder completo
        input_layer = tf.keras.Input(shape=(input_shape,))
        encoded = self.encoder(input_layer)
        decoded = self.decoder(encoded)
        self.autoencoder = tf.keras.models.Model(inputs=input_layer, outputs=decoded)

        # Compila el autoencoder
        self.autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

    def fit(self, X, y=None, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        self.autoencoder.fit(X, X, epochs=10, batch_size=32, sample_weight=sample_weight)

    def get_encoded_data(self, X):
        # TODO: devuelve la salida del encoder (code)
        return self.encoder.predict(X)

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session()  # Necesario para liberar la memoria en GPU


## Crea tu propio Clasificador

A continuación crearemos un clasificador que sea capaz de predecir la etiqueta pero no a partir de los datos originales, sino de la codificación `code` aprendida por el autoencoder. El diseño del clasificador es libre, pero recuerda que tiene que ser simple (máximo dos capas). **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [ ]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self):
        # TODO : define el modelo y compílalo

        # Definimos un clasificador sencillo (input -> dense -> output)
        self.model = tf.keras.models.Sequential([
            tf.keras.layers.InputLayer(input_shape=(64,)),   # Asumiendo que el code tiene tamaño 64
            tf.keras.layers.Dense(32, activation='relu'),    # Primera capa oculta (simple)
            tf.keras.layers.Dense(10, activation='softmax')  # Salida con softmax para 10 clases (como MNIST)
        ])

        # Compila el modelo
        self.model.compile(optimizer='adam',
                           loss='sparse_categorical_crossentropy',
                           metrics=['accuracy'])

    def fit(self, X, y, sample_weight=None):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras
        self.model.fit(X, y, epochs=10, batch_size=32, sample_weight=sample_weight)

    def predict(self, X):
        # TODO: devuelve la clase ganadora
        return self.model.predict(X).argmax(axis=1)

    def predict_proba(self, X):
        # TODO: devuelve las probabilidades de cada clase
        return self.model.predict(X)

    def score(self, X, y):
        # TODO: calcula la precisión
        _, accuracy = self.model.evaluate(X, y, verbose=0)
        return accuracy

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session()  # Necesario para liberar la memoria en GPU


### Entrenamiendo del modelo sin supervisar

Primero de todo, a modo de comparación, crea un modelo (de capacidad similar a tu encoder+clasificador) que puedas entrenar supervisadamente con `x_train` e `y_train`. Anota su rendimiento.

In [ ]:
# TODO - Crea un modelo, entrénalo con x_train e y_train y muestra su rendimiento en test.

# Definir el modelo supervisado
supervised_model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(784,)),      # Imágenes aplanadas
    tf.keras.layers.Dense(128, activation='relu'),        # Capa oculta 1
    tf.keras.layers.Dense(64, activation='relu'),         # Capa oculta 2
    tf.keras.layers.Dense(32, activation='relu'),         # Capa oculta 3
    tf.keras.layers.Dense(10, activation='softmax')       # Capa de salida
])

# Compilar el modelo
supervised_model.compile(optimizer='adam',
                         loss='sparse_categorical_crossentropy',
                         metrics=['accuracy'])

# Entrenar el modelo
supervised_model.fit(x_train, y_train, epochs=10, batch_size=32)

# Evaluar en test
test_loss, test_accuracy = supervised_model.evaluate(x_test, y_test)

print(f"Test Accuracy: {test_accuracy:.4f}")

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2181 - loss: 2.1602
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5572 - loss: 1.4390
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7971 - loss: 0.8166
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8607 - loss: 0.5297
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9215 - loss: 0.3421 
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9503 - loss: 0.2632 
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9778 - loss: 0.1568
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9880 - loss: 0.1064
Epoch 9/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9860 - loss: 0.0961
Epoch 10/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9943 - loss: 0.0629
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8536 - loss: 0.5416
Test Accuracy: 0.8718


### Entrenamiendo del modelo semisupervisado

El entrenamiento del sistema semisupervisado se realiza en dos pasos.

1. Se entrena el autoencoder con todos los datos (etiquetados y sin etiquetar).
1. Se entrena un clasificador simple (una o dos capas), teniendo como entrada la salida del encoder (**code**) de los datos etiquetados.

<font color='red'>NOTA:</font> para entrenar (y predecir) vamos a utilizar las funciones que hemos definido en el autoencoder y en el clasificador.

In [ ]:
# TODO: implementa el algoritmo semisupervised_training.

def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):
    # Paso 1: Entrenar el autoencoder con datos etiquetados + no etiquetados
    all_data = np.vstack((x_train, unlabeled_data))  # Concatenar todos los datos
    autoencoder.fit(all_data)

    # Paso 2a: Codificar los datos etiquetados
    encoded_x_train = autoencoder.get_encoded_data(x_train)

    # Paso 2b: Entrenar el clasificador con las codificaciones
    classifier.fit(encoded_x_train, y_train)

### Entrenamos nuestro modelo

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada.

In [ ]:
# Crea tu autoencoder y tu clasificador
autoencoder = MiAutoencoder(input_shape=x_train.shape[1])
classifier = MiClasificador()

In [ ]:
# TODO: Entrena tu modelo
semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_train)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.1788
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.0922
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.0842
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.0812
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0790
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0776
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0765
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.0757
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.0750
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0746
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0999 - loss: 6.2269   
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1510 - loss: 3.3109 
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2169 - loss: 2.4449 
Epoch 4

In [ ]:
# TODO: Obtén la precisión sobre el conjunto de test
pred_data = autoencoder.get_encoded_data(x_test)
print('Test accuracy :', classifier.score(pred_data, y_test))

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Test accuracy : 0.5127999782562256


## Mejorando el código

Nuestro modelo actual requiere de dos pasos para entrenarse, pero podría realizarse en un único paso si **creamos un modelo con las dos salidas (autoencoder y clasificador)**.

Para ello, hay que tener en cuenta que, en los datos sin etiquetar, su contribución al clasificador debería ser nula.


### TRABAJO: Crea el nuevo modelo y modifica la función semisupervised_training para tener en cuenta todos los puntos mencionados anteriormente

In [ ]:
# TODO: crea el nuevo modelo

# TODO: crea tu propio clasificador

class MiClasificadorSemisupervisado:

    def __init__(self, input_shape):
        # TODO : define el modelo y compílalo

        # Definimos la entrada del modelo
        inputs = tf.keras.Input(shape=(input_shape,))

        # Encoder: primera capa densa con 128 neuronas y activación ReLU
        x = tf.keras.layers.Dense(128, activation='relu')(inputs)
        # Encoder: segunda capa densa con 64 neuronas (código aprendido)
        code = tf.keras.layers.Dense(64, activation='relu')(x)

        # Decoder: primera capa densa de reconstrucción
        x_decoded = tf.keras.layers.Dense(128, activation='relu')(code)
        # Decoder: capa final de salida para reconstruir la entrada (activación sigmoide)
        decoded_output = tf.keras.layers.Dense(input_shape, activation='sigmoid', name='decoder_output')(x_decoded)

        # Clasificador: primera capa para clasificación
        x_classif = tf.keras.layers.Dense(32, activation='relu')(code)
        # Clasificador: capa de salida para clasificación en 10 clases
        class_output = tf.keras.layers.Dense(10, activation='softmax', name='classifier_output')(x_classif)

        # Crear el modelo con dos salidas: reconstrucción y clasificación
        self.model = tf.keras.Model(inputs=inputs, outputs=[decoded_output, class_output])

        # Compilar el modelo especificando pérdidas y métricas para cada salida
        self.model.compile(optimizer='adam',
                           loss={
                               'decoder_output': 'binary_crossentropy',
                               'classifier_output': 'sparse_categorical_crossentropy'
                           },
                           metrics={
                               'classifier_output': 'accuracy'
                           })

    def fit(self, X, y, unlabeled_data):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras, y define bien el sample_weight

        # Combinar datos etiquetados y no etiquetados verticalmente
        all_data = np.vstack((X, unlabeled_data))

        # Etiquetas para datos etiquetados
        y_labeled = y
        # Etiquetas falsas (ceros) para los datos no etiquetados
        y_unlabeled = np.zeros((unlabeled_data.shape[0],))

        # Concatenar etiquetas reales y ficticias
        all_labels = np.concatenate((y_labeled, y_unlabeled))

        # Asignar pesos 1 a todos los ejemplos para el decoder (reconstrucción)
        decoder_sample_weight = np.ones((all_data.shape[0],))
        # Asignar peso 1 a datos etiquetados y 0 a no etiquetados para el clasificador
        classifier_sample_weight = np.concatenate((
            np.ones((X.shape[0],)),
            np.zeros((unlabeled_data.shape[0],))
        ))

        # Entrenar el modelo con las salidas y pesos definidos
        self.model.fit(
            all_data,
            {'decoder_output': all_data, 'classifier_output': all_labels},
            sample_weight={'decoder_output': decoder_sample_weight,
                           'classifier_output': classifier_sample_weight},
            epochs=10,
            batch_size=32
        )

    def predict(self, X):
        # TODO: devuelve la clase ganadora del clasificador
        decoded_output, class_output = self.model.predict(X)
        return np.argmax(class_output, axis=1)  # Seleccionar la clase con mayor probabilidad

    def predict_proba(self, X):
        # TODO: devuelve la probabilidad del clasificador
        decoded_output, class_output = self.model.predict(X)
        return class_output  # Devolver directamente las probabilidades de cada clase

    def score(self, X, y):
        # TODO: calcula la precisión
        decoded_output, class_output = self.model.predict(X)
        predictions = np.argmax(class_output, axis=1)  # Convertir predicciones en etiquetas
        accuracy = np.mean(predictions == y)  # Calcular porcentaje de aciertos
        return accuracy

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session()  # Liberar memoria en GPU


In [ ]:
# TODO: reescribe la función semisupervised_training para incorporar las mejoras mencionadas anteriormente

def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    # Combinar los datos etiquetados + no etiquetados
    all_data = np.vstack((x_train, unlabeled_data))

    # Crear etiquetas para el clasificador
    y_labeled = y_train  # Etiquetas reales
    y_unlabeled = np.zeros((unlabeled_data.shape[0],))  # Etiquetas ficticias para datos no etiquetados

    # Concatenar todas las etiquetas
    all_labels = np.concatenate((y_labeled, y_unlabeled))

    # Crear etiquetas esperadas para el modelo como lista de salidas
    labels = [all_data, all_labels]

    # Crear sample_weight para el decoder (todos tienen peso 1)
    decoder_sample_weight = np.ones((all_data.shape[0],))
    # Crear sample_weight para el clasificador (solo etiquetados tienen peso 1)
    classifier_sample_weight = np.concatenate((
        np.ones((x_train.shape[0],)),
        np.zeros((unlabeled_data.shape[0],))
    ))

    # Agrupar los sample_weights para las dos salidas
    sample_weights = [decoder_sample_weight, classifier_sample_weight]

    # Llamar a fit() del modelo pasando los datos, etiquetas y pesos
    model.model.fit(
        all_data,
        labels,
        sample_weight=sample_weights,
        epochs=10,
        batch_size=32
    )


In [ ]:
# TODO: Crea y entrena tu clasificador
model = MiClasificadorSemisupervisado(input_shape=x_train.shape[1])

semisupervised_training_v2(model, x_train, y_train, unlabeled_train)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - classifier_output_accuracy: 0.0603 - classifier_output_loss: 0.0187 - decoder_output_loss: 0.1980 - loss: 0.2167
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - classifier_output_accuracy: 0.1178 - classifier_output_loss: 0.0089 - decoder_output_loss: 0.1059 - loss: 0.1148
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - classifier_output_accuracy: 0.1050 - classifier_output_loss: 0.0056 - decoder_output_loss: 0.0945 - loss: 0.1001
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - classifier_output_accuracy: 0.1081 - classifier_output_loss: 0.0041 - decoder_output_loss: 0.0904 - loss: 0.0945
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - classifier_output_accuracy: 0.1088 - classifier_output_loss: 0.0048 - decoder_output_loss: 0.0876 - loss: 0.0924
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - classifier_output_accuracy: 0.1034 - classifier_output_loss: 0.0029 - decoder_output_loss: 0.

In [ ]:
# TODO: Obtén la precisión sobre el conjunto de test
print('Test accuracy :', model.score(x_test, y_test))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Test accuracy : 0.7732


# Hay vida más allá del autoencoder

*   Elemento de lista
*   Elemento de lista



¿Has probado a utilizar otro método distinto del autoencoder para obtener una respresentación similar a la salida del encoder? La idea es la siguiente:

1. Define un modelo $model$ convolucional similar al encoder de un autoencoder (la entrada es el tamaño de la imagen, la salida el vector de representación)
1. Define una capa de salida $cluster$ que, partiendo de la salida de model, nos devuelva una salida con el mismo número de clases que el dataset a utilizar (la entrada es el vector de representación), usando softmax como activación de salida
1. Para cada batch de entrenamiento $X$:  # Usa un batch alto, mínimo 128
  1. Modifica las imágenes de entrada con [data_augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_1$.
  1. Modifica otra vez las imágenes de entrada con [data_augmentation_2](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_2$.
  1. $augX_{1comp} \leftarrow model(augX_1)$
  1. $augX_{2comp} \leftarrow model(augX_2)$
  1. $cX_{1comp} \leftarrow cluster(augX_{1comp})$
  1. $cX_{2comp} \leftarrow cluster(augX_{2comp})$
  1. $M \leftarrow augX_{1comp} ~ augX_{2comp}^T$
  1. $loss_C \leftarrow cX_{1comp}(1 - cX_{1comp}) + cX_{2comp}(1 - cX_{2comp})$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
  1. $loss_M \leftarrow crossentropy(I, softmax(M/\tau, axis=1)))$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
    1. $\tau$ es un hiperparámetro que se suele definir a 5.0
  1. $loss \leftarrow loss_M + \lambda~loss_C$
    1. $\lambda$ es un hiperparámetro (puedes probar con 0.5)


In [12]:
# Escribe aquí la solución. Crea tantos bloques de código como necesites. Puedes utilizar la siguiente red para generar distorsiones

data_augmentation = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(48, 48), # para CIFAR, para MNIST usar 40 en lugar de 48
        tf.keras.layers.RandomCrop(32, 32), # para CIFAR, para MNIST usar 28 en lugar de 32
    ]
)


**Preparamos datos para trabajar con convolución**

In [2]:
# TODO: Preparamos los datos para trabajar con convolucionales

labeled_data = 0.01  # Vamos a usar solo el 1% de los datos etiquetados
np.random.seed(42)

# Cargamos MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Creamos índices aleatorios para separar datos etiquetados y no etiquetados
indices = np.arange(len(x_train))
np.random.shuffle(indices)
ntrain_data = int(labeled_data * len(x_train))

# Separar datos
unlabeled_train = x_train[indices[ntrain_data:]]  # Datos no etiquetados
x_train = x_train[indices[:ntrain_data]]           # Datos etiquetados
y_train = y_train[indices[:ntrain_data]]

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# Normalizamos todos los datos al rango [0,1]
x_train = x_train.astype(np.float32) / 255.0
unlabeled_train = unlabeled_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Expandimos el canal (añadimos dimensión extra)
# MNIST original está como (28,28), queremos (28,28,1) para Conv2D
x_train = np.expand_dims(x_train, axis=-1)
unlabeled_train = np.expand_dims(unlabeled_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print(x_train.shape, unlabeled_train.shape, x_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
x_train shape: (600, 28, 28)
600 train samples
10000 test samples
(600, 28, 28, 1) (59400, 28, 28, 1) (10000, 28, 28, 1)


## Bloque 1: Definir model y cluster

In [3]:
# TODO: Definimos el modelo convolucional "encoder" y la capa "cluster" final

# Definimos el encoder (modelo convolucional)
def create_encoder(input_shape=(28, 28, 1), code_size=64):
    # Creamos un pequeño modelo convolucional
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)  # Primera capa conv
    x = tf.keras.layers.MaxPooling2D()(x)                                          # Pooling
    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)         # Segunda capa conv
    x = tf.keras.layers.MaxPooling2D()(x)                                           # Pooling

    x = tf.keras.layers.Flatten()(x)                                                # Aplanar
    code = tf.keras.layers.Dense(code_size, activation='relu')(x)                  # Código aprendido

    model = tf.keras.Model(inputs, code, name='encoder')
    return model

# Definimos la capa de clustering
def create_cluster_layer(input_dim, n_clusters=10):
    # Capa densa con activación softmax (salida de clustering)
    return tf.keras.layers.Dense(n_clusters, activation='softmax', name='cluster_output')

# Creamos el encoder
encoder = create_encoder()

# Creamos la capa de clustering
cluster_layer = create_cluster_layer(input_dim=64)

## Bloque 2: Preparar un train_step para un batch

In [4]:
# TODO: Definimos un paso de entrenamiento personalizado para un batch

# Definimos algunos hiperparámetros
n_clusters = 10      # Número de clusters
tau = 5.0            # Hiperparámetro de suavizado para softmax de M
lambda_ = 0.5        # Peso para combinar las pérdidas


# Inicializamos el data augmentation que nos dieron
data_augmentation = tf.keras.models.Sequential(
    [
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(40, 40), # Para MNIST
        tf.keras.layers.RandomCrop(28, 28), # Para MNIST
    ]
)

# Definimos el optimizador
optimizer = tf.keras.optimizers.Adam()

# Función para un solo paso de entrenamiento
@tf.function
def train_step(X_batch):
    # X_batch: batch de imágenes originales

    # 1. Aplicamos dos augmentaciones diferentes
    augX1 = data_augmentation(X_batch)      # Primera versión aumentada
    augX2 = data_augmentation_2(X_batch)     # Segunda versión aumentada

    with tf.GradientTape() as tape:
        # 2. Pasamos ambas por el encoder
        augX1_comp = encoder(augX1)
        augX2_comp = encoder(augX2)

        # 3. Obtenemos las asignaciones de cluster para cada una
        cX1_comp = cluster_layer(augX1_comp)
        cX2_comp = cluster_layer(augX2_comp)

        # 4. Calculamos la matriz M
        # Multiplicamos augX1_comp por augX2_comp^T
        M = tf.linalg.matmul(augX1_comp, augX2_comp, transpose_b=True)  # Matriz de similitudes

        # 5. Definimos las dos pérdidas

        # Primera pérdida: auto-coherencia entre clusters (contrastive)
        loss_c = tf.reduce_mean(
            tf.reduce_sum(cX1_comp * (1 - cX1_comp), axis=1) +
            tf.reduce_sum(cX2_comp * (1 - cX2_comp), axis=1)
        )

        # Segunda pérdida: cross-entropy sobre M suavizada
        M_softmax = tf.nn.softmax(M / tau, axis=1)
        loss_M = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(
            y_true=tf.nn.softmax(M / tau, axis=1),
            y_pred=M_softmax,
            from_logits=False
        ))

        # 6. Combinamos las dos pérdidas
        loss = loss_M + lambda_ * loss_c

    # 7. Calculamos y aplicamos gradientes
    grads = tape.gradient(loss, encoder.trainable_variables + cluster_layer.trainable_variables)
    optimizer.apply_gradients(zip(grads, encoder.trainable_variables + cluster_layer.trainable_variables))

    return loss


## Bloque 3: Entrenamiento completo (recorrer batches)

In [5]:
# TODO: Definimos el bucle completo de entrenamiento

# Hiperparámetros de entrenamiento
batch_size = 256        # Batch alto como se pide (puedes ajustar)
epochs = 20             # Número de épocas de entrenamiento

# Importante: Usamos unlabeled_train porque el método no supervisado no usa etiquetas
# Creamos el tf.data.Dataset para los datos no etiquetados
train_dataset = tf.data.Dataset.from_tensor_slices(unlabeled_train)  # Usamos solo datos no etiquetados
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)
# .shuffle mezcla los datos
# .batch agrupa los datos en lotes
# .prefetch mejora el rendimiento de lectura

# Bucle de entrenamiento
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")

    total_loss = 0.0  # Acumula la pérdida total de la época
    num_batches = 0   # Cuenta el número de batches

    for batch in train_dataset:
        loss = train_step(batch)  # Aplicamos un paso de entrenamiento
        total_loss += loss
        num_batches += 1

    # Calculamos y mostramos la pérdida promedio de la época
    avg_loss = total_loss / num_batches
    print(f"  Loss: {avg_loss:.4f}")


Epoch 1/20
  Loss: 0.2254
Epoch 2/20
  Loss: 0.0044
Epoch 3/20
  Loss: 0.0057
Epoch 4/20
  Loss: 0.0033
Epoch 5/20
  Loss: 0.0041
Epoch 6/20
  Loss: 0.0020
Epoch 7/20
  Loss: 0.0106
Epoch 8/20
  Loss: 0.0024
Epoch 9/20
  Loss: 0.0007
Epoch 10/20
  Loss: 0.0029
Epoch 11/20
  Loss: 0.0003
Epoch 12/20
  Loss: 0.0001
Epoch 13/20
  Loss: 0.0003
Epoch 14/20
  Loss: 0.0007
Epoch 15/20
  Loss: 0.0002
Epoch 16/20
  Loss: 0.0000
Epoch 17/20
  Loss: 0.0002
Epoch 18/20
  Loss: 0.0003
Epoch 19/20
  Loss: 0.0003
Epoch 20/20
  Loss: 0.0004


## Bloque 4: Extraer representaciones (features) del encoder

In [6]:
# TODO: Extraemos las codificaciones (features) usando el encoder entrenado

# Extraemos features del conjunto de datos de test
features_test = encoder(x_test, training=False)  # ¡Importante usar training=False!

print('Shape de features extraídos:', features_test.shape)

Shape de features extraídos: (10000, 64)


## Bloque 5: Agrupar por KMeans y evaluar

In [8]:
# TODO: Aplicar clustering sobre las codificaciones extraídas

from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment
import numpy as np

# Aplicamos KMeans con 10 clusters
kmeans = KMeans(n_clusters=10, random_state=42)
pred_clusters = kmeans.fit_predict(features_test.numpy())

# Para comparar con las etiquetas reales necesitamos resolver el problema de asignación óptima (permuta clusters)
def cluster_accuracy(y_true, y_pred):
    # Matriz de confusión
    D = tf.math.confusion_matrix(y_true, y_pred).numpy()
    # Resolver problema de asignación (Hungarian algorithm)
    row_ind, col_ind = linear_sum_assignment(-D)
    # Calcular la precisión
    return D[row_ind, col_ind].sum() / D.sum()

# Calculamos la precisión
clustering_acc = cluster_accuracy(y_test, pred_clusters)

print(f"Precisión de clustering (sin etiquetas): {clustering_acc:.4f}")

Precisión de clustering (sin etiquetas): 0.2405


# ¡ENHORABUENA! Has completado la práctica de modelos generativos.


# Trabajo extra

¿Has probado a hacer el autoencoder totalmente convolucional? Para el *decoder* puedes usar las funciones [UpSampling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/UpSampling2D) o [Conv2DTranspose](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose).